# Notebook 01 — Data Preprocessing

Replicates MATLAB `step1_importDataToMatlab.m` and `step2_preprocessData.m`.

Each dataset is loaded from CSV, and subjects are excluded if they fail any of three criteria:
1. **Accuracy** outside [60%, 95%]
2. **Response stereotypy** — >85% of responses are the same
3. **Confidence stereotypy** — >85% of confidence ratings are the same

Two dataset-specific subtleties must be handled to match MATLAB exactly:
- **Maniscalco**: NaN (missed) responses count as *incorrect* before computing accuracy
- **Rouault Expt 1**: confidence stereotypy is checked on *raw* ratings (1–11) *before* the `conf − 5, clip ≥ 1` transformation

In [1]:
import sys, os, warnings
warnings.filterwarnings('ignore')

# ── locate repository root and add src/ to path ──────────────
REPO = os.path.abspath(os.path.join(
    os.getcwd(), '..' if os.path.basename(os.getcwd()) == 'notebooks' else '.'))
DATA = os.path.join(REPO, 'matlab', 'metasignal_mat', 'Preprocess', 'orig_csv_files')
OUT  = os.path.join(REPO, 'notebooks', 'precomputed')
sys.path.insert(0, os.path.join(REPO, 'src'))
os.makedirs(OUT, exist_ok=True)

import numpy as np
import pandas as pd
from scipy import stats
from metasignal.stdpy.compute_all import compute_all_measures
print("metasignal loaded successfully.")


metasignal loaded successfully.


## Helper: subject exclusion

In [2]:
ACC_LO, ACC_HI, MAX_PROP = 0.60, 0.95, 0.85

# ── exclusion thresholds (match MATLAB step2_preprocessData.m) ──
ACC_LO, ACC_HI, MAX_PROP = 0.60, 0.95, 0.85

def _exclude(acc, pr, pc):
    """Return True if subject should be excluded."""
    return acc < ACC_LO or acc > ACC_HI or pr > MAX_PROP or pc > MAX_PROP

def preprocess_haddara():
    """Load Haddara 2022 Expt2. n_ratings=4. Returns list of subject dicts."""
    df = pd.read_csv(os.path.join(DATA, 'data_Haddara_2022_Expt2.csv'))
    subjects = []
    for sid, g in df.groupby('Subj_idx'):
        g = g.dropna(subset=['Stimulus','Response','Confidence'])
        stim = g['Stimulus'].to_numpy(float)
        resp = g['Response'].to_numpy(float)
        conf = g['Confidence'].to_numpy(float)
        day  = g['Day'].to_numpy(float)
        acc = np.mean(stim == resp)
        pr  = np.max(np.unique(resp, return_counts=True)[1]) / len(resp)
        pc  = np.max(np.unique(conf, return_counts=True)[1]) / len(conf)
        if _exclude(acc, pr, pc): continue
        subjects.append(dict(sid=sid, stim=stim, resp=resp, conf=conf, day=day, n_ratings=4))
    return subjects

def preprocess_maniscalco():
    """Load Maniscalco 2017 Expt1. NaN responses counted as incorrect. n_ratings=4."""
    df = pd.read_csv(os.path.join(DATA, 'data_Maniscalco_2017_expt1.csv'))
    subjects = []
    for sid, g in df.groupby('Subj_idx'):
        stim_all = g['Stimulus'].to_numpy(float)
        resp_all = g['Response'].to_numpy(float)
        conf_all = g['Confidence'].to_numpy(float)
        # MATLAB: NaN responses count as incorrect
        correct = np.where(np.isnan(resp_all), 0.0, (stim_all == resp_all).astype(float))
        acc = np.mean(correct)
        pr  = np.max(np.bincount(resp_all[~np.isnan(resp_all)].astype(int))) / len(resp_all)
        valid_c = conf_all[~np.isnan(conf_all)]
        pc  = np.max(np.bincount(valid_c.astype(int))) / len(conf_all)
        if _exclude(acc, pr, pc): continue
        ok = ~(np.isnan(stim_all) | np.isnan(resp_all) | np.isnan(conf_all))
        subjects.append(dict(sid=sid, stim=stim_all[ok], resp=resp_all[ok],
                             conf=conf_all[ok], n_ratings=4))
    return subjects

def preprocess_shekhar():
    """Load Shekhar 2021. Continuous conf (50–100) binned to n_ratings=6 levels."""
    df = pd.read_csv(os.path.join(DATA, 'data_Shekhar_2021.csv'))
    n_ratings = 6
    edges = np.linspace(50, 100, n_ratings + 1)
    subjects = []
    for sid, g in df.groupby('Subj_idx'):
        stim     = g['Stimulus'].to_numpy(float)
        resp     = g['Response'].to_numpy(float)
        conf_raw = g['Confidence'].to_numpy(float)
        contrast = g['Contrast'].to_numpy(float)
        conf = np.clip(np.digitize(conf_raw, edges, right=True), 1, n_ratings)
        acc = np.mean(stim == resp)
        pr  = np.max(np.unique(resp, return_counts=True)[1]) / len(resp)
        pc  = np.max(np.unique(conf, return_counts=True)[1]) / len(conf)
        if _exclude(acc, pr, pc): continue
        subjects.append(dict(sid=sid, stim=stim, resp=resp, conf=conf,
                             contrast=contrast, n_ratings=n_ratings))
    return subjects

def preprocess_rouault(expt=1):
    """Load Rouault 2018 Expt1 or Expt2. Stereotypy on raw conf; transform after. n_ratings=6."""
    df = pd.read_csv(os.path.join(DATA, f'data_Rouault_2018_Expt{expt}.csv'))
    subjects = []
    for sid, g in df.groupby('Subj_idx'):
        g = g.dropna(subset=['Stimulus','Response','Confidence'])
        stim     = g['Stimulus'].to_numpy(float)
        resp     = g['Response'].to_numpy(float)
        conf_raw = g['Confidence'].to_numpy(float)
        dotdiff  = g['DotDiff'].to_numpy(float)
        acc = np.mean(stim == resp)
        pr  = np.max(np.bincount(resp.astype(int))) / len(resp)
        # MATLAB: stereotypy on RAW conf (1-11) BEFORE transformation
        pc  = np.max(np.bincount(conf_raw.astype(int))) / len(conf_raw)
        if _exclude(acc, pr, pc): continue
        conf = np.clip(conf_raw - 5, 1, None) if expt == 1 else conf_raw.copy()
        subjects.append(dict(sid=sid, stim=stim, resp=resp, conf=conf,
                             contrast=dotdiff, n_ratings=6))
    return subjects

def preprocess_locke():
    """Load Locke 2020 (test trials only). n_ratings=2."""
    df = pd.read_csv(os.path.join(DATA, 'data_Locke_2020.csv'))
    df = df[df['Training'] == 0].copy()
    df['Confidence'] = df['Confidence'] + 1   # 0/1 → 1/2
    subjects = []
    for sid, g in df.groupby('Subj_idx'):
        g = g.dropna(subset=['Stimulus','Response','Confidence'])
        stim      = g['Stimulus'].to_numpy(float)
        resp      = g['Response'].to_numpy(float)
        conf      = g['Confidence'].to_numpy(float)
        condition = g['Condition'].to_numpy(float)
        acc = np.mean(stim == resp)
        pr  = np.max(np.unique(resp, return_counts=True)[1]) / len(resp)
        if acc < ACC_LO or acc > ACC_HI or pr > MAX_PROP: continue
        subjects.append(dict(sid=sid, stim=stim, resp=resp, conf=conf,
                             condition=condition, n_ratings=2))
    return subjects

print("Preprocessing functions defined.")


Preprocessing functions defined.


## Load all datasets

In [3]:
haddara    = preprocess_haddara()
maniscalco = preprocess_maniscalco()
shekhar    = preprocess_shekhar()
rouault1   = preprocess_rouault(1)
rouault2   = preprocess_rouault(2)
locke      = preprocess_locke()

expected = dict(Haddara=70, Maniscalco=22, Shekhar=20,
                Rouault1=466, Rouault2=484, Locke=10)
actual   = dict(Haddara=len(haddara), Maniscalco=len(maniscalco),
                Shekhar=len(shekhar), Rouault1=len(rouault1),
                Rouault2=len(rouault2), Locke=len(locke))

print(f"{'Dataset':<15} {'Expected':>10} {'Got':>6} {'Match':>7}")
print("-"*42)
for k in expected:
    ok = "✓" if expected[k] == actual[k] else "✗"
    print(f"{k:<15} {expected[k]:>10} {actual[k]:>6} {ok:>7}")


Dataset           Expected    Got   Match
------------------------------------------
Haddara                 70     70       ✓
Maniscalco              22     22       ✓
Shekhar                 20     20       ✓
Rouault1               466    466       ✓
Rouault2               484    484       ✓
Locke                   10     10       ✓


## Per-dataset structure

Each subject is stored as a dict with keys:
`stim`, `resp`, `conf` (NumPy arrays), `n_ratings` (int), plus dataset-specific extras.

In [4]:
s = haddara[0]
print("Haddara subject 0:")
print(f"  Trials : {len(s['stim'])}")
print(f"  n_ratings: {s['n_ratings']}")
print(f"  Accuracy : {np.mean(s['stim']==s['resp']):.3f}")
print(f"  Conf range: {s['conf'].min():.0f} – {s['conf'].max():.0f}")
print(f"  Extra keys: {[k for k in s if k not in ('stim','resp','conf','n_ratings','sid')]}")


Haddara subject 0:
  Trials : 3350
  n_ratings: 4
  Accuracy : 0.738
  Conf range: 1 – 4
  Extra keys: ['day']
